# NatDrug-AI v2 — Complete automatic study (EGFR / CHEMBL203)

**Run All Cells** to check source files, reuse validated model results, benchmark additional models, calibrate predictions on held-out ChEMBL scaffolds, screen the real IMPPAT natural-product library, and generate scientific reports. This notebook **will NOT fabricate natural-product validation metrics** if original experimental provenance is unavailable.

**Scientific distinction:** internal ChEMBL scaffold validation ≠ independently verified natural-product EGFR validation. Screening predictions ≠ experimental inhibition. This package includes genuine source records and prior Mac outputs.

## Step 1 — Find your project and verify the Python environment
Run this cell before installing anything. It uses the exact Python kernel currently running Jupyter, not your Mac system Python.

In [ ]:
from pathlib import Path
import sys, platform, importlib.util
import pandas as pd

cwd=Path.cwd().resolve()
ROOT=next((p for p in [cwd,*cwd.parents] if (p/'data/CHEMBL203_IC50.csv').exists() and (p/'code/v2_methods.py').exists()),None)
if ROOT is None:
    raise FileNotFoundError('Could not find NatDrugAI_v2_Automated_Jupyter. Open this notebook from within the extracted project folder.')
print('Project:',ROOT)
print('Jupyter Python:',sys.executable)
print('OS:',platform.platform())
for lib in ('rdkit','sklearn','numpy','pandas','matplotlib','requests','torch','tabulate'):
    print(f'{lib:<12}', 'READY' if importlib.util.find_spec(lib) else 'NOT INSTALLED')
print('Training file exists:',(ROOT/'data/CHEMBL203_IC50.csv').exists())
print('IMPPAT chemical structures:',(ROOT/'sources/imppat_chemical_normalized.csv').exists())
print('IMPPAT bioactivity records:',(ROOT/'sources/imppat_bioactivity.tsv').exists())

**Only if dependencies are missing**, create a new code cell and run:

```python
%pip install -r ../code/requirements_v2.txt
```

Then restart the Jupyter kernel. For rerunning MLP also install `torch` in your selected kernel. No need to reinstall if all packages are already present.

## Step 2 — Settings

Cached source-backed benchmark and robustness results are reused. Set flags to `True` to recompute (this takes longer). By default, the additional model comparison only runs when its output is absent. The existing source-audited screening is included, and a separate scaffold-calibrated screening is available.

In [ ]:
# ------- EDIT THESE FLAGS IF NEEDED -------
FORCE_REBUILD_BENCHMARK = False      # Existing 5-fold measured ChEMBL study is included
FORCE_REBUILD_IMPPAT_PREP = False    # Existing 18,030-molecule IMPPAT preparation is included
FORCE_REBUILD_ROBUSTNESS = False     # Existing 3-seed MLP + curation analysis included
RUN_ADDITIONAL_MODEL_COMPARISON = True  # ExtraTrees scaffold CV; can be slow
RUN_UNCERTAINTY_SCREENING = True        # New RF+descriptors, scaffold intervals + 18,030 NPs
FORCE_REBUILD_UNCERTAINTY = False
FETCH_TRAINING_ASSAY_METADATA_FROM_CHEMBL = False  # Optional online requests to 477 assays
ASSAY_FETCH_LIMIT = None  # None means all when enabled
VERIFIED_PRIMARY_RECORDS_CSV = None  # e.g. ROOT/'results/v2/provenance/provenance_review_COMPLETED.csv'

print('Flags configured. Reusing prior Mac results unless FORCE flags are enabled.')

## Step 3 — Reproducible execution helper and input audit

Scripts write real CSVs, figures, JSON source manifests and detailed text logs. If any stage fails the notebook displays the actual error rather than reporting it as successful.

In [ ]:
import subprocess, time, json, hashlib, os, zipfile
from IPython.display import display, Markdown, Image
from datetime import datetime, timezone
LOGDIR=ROOT/'results/v2/logs';LOGDIR.mkdir(parents=True,exist_ok=True)

def run_script(label, filename, args=(), skip=False):
    if skip:
        print(f'REUSED CACHED OUTPUT: {label}')
        return True
    command=[sys.executable,'-u',str(ROOT/'code'/filename),*map(str,args)]
    print('\nRUN:',label,'\n',' '.join(command),flush=True)
    log=LOGDIR/(label.replace(' ','_')+'.log')
    with log.open('w',encoding='utf8') as fp:
        p=subprocess.Popen(command,cwd=str(ROOT),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                           text=True,bufsize=1)
        for line in p.stdout:
            print(line.rstrip(),flush=True);fp.write(line);fp.flush()
        exitcode=p.wait()
    if exitcode:
        raise RuntimeError(f'{label} failed (exit {exitcode}). See {log}')
    print('SUCCESS:',label,flush=True)
    return True

from importlib.machinery import SourceFileLoader
sys.path.insert(0,str(ROOT/'code'))
from v2_methods import file_info
print(json.dumps(file_info(ROOT),indent=2))
print('Source hashes and software versions are retained in the final manifest.')

## Step 4 — Experimental training benchmark (Experiment 1)

Uses the genuine **4,991 ChEMBL activity rows** and the earlier measured 5-fold results, or recomputes on demand. It compares random/scaffold splits and simple baselines. Never use 13 external IMPPAT compounds for model selection.

In [ ]:
benchmark=ROOT/'results/legacy_benchmark/summary_metrics.csv'
run_script('01_CHEmBL203_benchmark','benchmark_natdrug_ai.py',[
    '--csv',ROOT/'data/CHEMBL203_IC50.csv','--out',ROOT/'results/legacy_benchmark',
    '--target','CHEMBL203'],skip=(benchmark.exists() and not FORCE_REBUILD_BENCHMARK))
display(pd.read_csv(benchmark).round(4))

## Step 5 — IMPPAT experimental source data and chemical library

Reuses the original normalized IMPPAT chemical export and bioactivity table. Source database tags do **not** establish the original measurement's independence.

In [ ]:
candidate_csv=ROOT/'results/legacy_imppat_audit/external_egfr_candidates_NOT_YET_VALIDATED.csv'
run_script('02_IMPPAT_data_preparation','prepare_imppat.py',[
  '--chemical',ROOT/'sources/imppat_chemical_normalized.csv',
  '--bioactivity',ROOT/'sources/imppat_bioactivity.tsv',
  '--train',ROOT/'data/CHEMBL203_IC50.csv',
  '--out',ROOT/'results/legacy_imppat_audit'],
  skip=(candidate_csv.exists() and not FORCE_REBUILD_IMPPAT_PREP))
manifest=json.loads((ROOT/'results/legacy_imppat_audit/preparation_manifest.json').read_text())
print('Real IMPPAT compounds after structure overlap filtering:',manifest.get('screening_after_overlap'))
print('EGFR IC50 candidates (NOT verified external set):',manifest.get('candidate_measured_egfr_unique_compounds'))

## Step 6 — Audit original assay provenance and publication overlap

The IMPPAT export contains database source labels but *not primary DOI/PMID or assay identifiers*. The validation gate cannot be passed using those labels alone. This step prepares a review template. Optional live ChEMBL lookup retrieves training assay metadata if you enable it.

In [ ]:
run_script('03_source_provenance_audit','v2_methods.py',['audit','--root',ROOT])
if FETCH_TRAINING_ASSAY_METADATA_FROM_CHEMBL:
    opts=['fetch_assays','--root',ROOT]
    if ASSAY_FETCH_LIMIT is not None:opts+=['--limit',ASSAY_FETCH_LIMIT]
    run_script('04_training_assay_lookup','v2_methods.py',opts)
else:print('Optional ChEMBL metadata requests skipped; enable above to fetch official training assay IDs.')
display(pd.read_csv(ROOT/'results/v2/provenance/natural_product_candidates_NEED_VERIFICATION.csv').head(13))
print('Review template:',ROOT/'results/v2/provenance/provenance_review_TEMPLATE.csv')

## Step 7 — Model comparison

Previous **Random Forest, Ridge, median and MLP** model results are included. An additional **ExtraTrees** model is compared using the same five scaffold folds. Baseline model tuning is not performed on the external natural-product candidates.

In [ ]:
if RUN_ADDITIONAL_MODEL_COMPARISON:
    run_script('05_additional_model_comparison','v2_methods.py',[
      'compare','--root',ROOT],skip=(ROOT/'results/v2/model_comparison/additional_model_metrics.csv').exists())
    display(pd.read_csv(ROOT/'results/v2/model_comparison/additional_model_metrics.csv').round(4))
else:print('ExtraTrees optional benchmark skipped; use existing RF/Ridge/median/MLP comparison.')

## Step 8 — Robustness: three MLP seeds + alternative assay processing (Experiment 4)

Reuses prior actual experiments by default; enable force to retrain. A dominant-assay subset contains far fewer molecules, so do not interpret its R² as a like-for-like improvement or decline.

In [ ]:
rob=ROOT/'results/legacy_robustness/mlp_3seed_summary.csv'
run_script('06_model_robustness','robustness_studies.py',[
    '--csv',ROOT/'data/CHEMBL203_IC50.csv',
    '--out',ROOT/'results/legacy_robustness','--mode','all'],
    skip=(rob.exists() and not FORCE_REBUILD_ROBUSTNESS))
display(pd.read_csv(rob).round(4))
display(pd.read_csv(ROOT/'results/legacy_robustness/curation_sensitivity_summary.csv').round(4))

## Step 9 — Calibrated intervals on ChEMBL scaffolds + complete IMPPAT screening (Experiment 3)

A fixed RF is fitted on development scaffolds, with separate calibration and untouched test scaffolds. The same trained model then predicts the **18,030 real natural products**. Report *observed held-out ChEMBL coverage*, not guaranteed natural-product coverage. Low Tanimoto values mark possible out-of-domain predictions.

In [ ]:
if RUN_UNCERTAINTY_SCREENING:
    rargs=['reliability','--root',ROOT]
    if FORCE_REBUILD_UNCERTAINTY:rargs.append('--force')
    run_script('07_uncertainty_aware_screen','v2_methods.py',rargs,
       skip=((ROOT/'results/v2/reliability_screening/natural_products_with_intervals.csv').exists() and not FORCE_REBUILD_UNCERTAINTY))
    summary=json.loads((ROOT/'results/v2/reliability_screening/reliability_manifest.json').read_text())
    print('Natural products predicted:',summary['library_n'])
    print('Scaffold-disjoint ChEMBL test empirical interval coverage:',round(summary['heldout_scaffold_empirical_coverage'],3))
    print('ChEMBL test R²:',round(summary['heldout_scaffold_metrics']['R2'],3))
    print('Percentage below heuristic Tanimoto 0.4:',round(100*summary['library_fraction_similarity_lt_0p4'],1),'%')
    x=pd.read_csv(ROOT/'results/v2/reliability_screening/natural_products_with_intervals.csv')
    display(x[['Name','Predicted_pIC50','Max_train_Tanimoto','Lower_pIC50','Upper_pIC50','Scaffold_seen_in_train']].head(15).round(3))
else:
    print('Using the earlier 18,030-molecule screening; recalibrated model and intervals skipped.')

## Step 10 — Independent external natural-product EGFR validation GATE (Experiment 2)

This step deliberately **refuses to publish independent R²** without assay/publication nonoverlap. If you obtain authoritative BindingDB/NPASS primary IDs, complete the CSV in `results/v2/provenance/` and provide it in `VERIFIED_PRIMARY_RECORDS_CSV`. Independent evidence still needs comparison against all training papers and assays.

In [ ]:
eargs=['external','--root',ROOT]
if VERIFIED_PRIMARY_RECORDS_CSV is not None:eargs.extend(['--verified-csv',VERIFIED_PRIMARY_RECORDS_CSV])
run_script('08_external_validation_provenance_gate','v2_methods.py',eargs)
status=json.loads((ROOT/'results/v2/external_validation/external_validation_status.json').read_text())
print(json.dumps(status,indent=2))
assert status['external_validation_metrics'] is None, 'Unexpected external validation claim: independently review primary-source provenance first.' 

## Step 11 — Manuscript tables, data provenance, figures and final ZIP

The report explicitly separates actual internal performance, untested screening predictions, and blocked external validation. Use the generated Markdown as the basis for revising your paper, not as a substitute for external experimental data.

In [ ]:
# Check the optional Markdown reporting dependency before running report.py
import importlib.util, subprocess, sys
if importlib.util.find_spec('tabulate') is None:
    print('Installing missing tabulate for pandas Markdown tables...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tabulate>=0.9'])
run_script('09_manuscript_report','v2_methods.py',['report','--root',ROOT])
from IPython.display import Markdown,display
p=ROOT/'results/v2/report/RESEARCH_SUMMARY.md'
display(Markdown(p.read_text()))

fileout=ROOT/'NatDrugAI_v2_All_Outputs.zip'
with zipfile.ZipFile(fileout,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for folder in ['results/v2','results/legacy_benchmark','results/legacy_robustness',
                   'results/legacy_screening','results/legacy_imppat_audit']:
        for f in (ROOT/folder).rglob('*'):
            if f.is_file():z.write(f,f.relative_to(ROOT))
print('FINAL RESULT ARCHIVE:',fileout)
print('File size MB:',round(fileout.stat().st_size/1e6,2))

## Step 12 — What you can safely say in the manuscript

**Supported:** 5-fold scaffold CV on measured ChEMBL IC50, MLP stability, assay-curation sensitivity, 18,030 IMPPAT in-silico rankings, prediction intervals evaluated on ChEMBL-held-out scaffolds.

**Not supported yet:** reliable predictive accuracy for genuinely independent natural-product EGFR measurements; biochemical hit confirmation; a validated 0.4 Tanimoto cut-off; guaranteed natural-product uncertainty coverage.

**Submit next:** review the source provenance file and obtain original per-assay identifiers/DOIs for prospective or externally sourced EGFR activities. This can be labor-intensive and cannot be solved by inventing or automatically trusting metadata absent from source files. Share the output ZIP for a manuscript update.